# SMTM Multi-Asset All-In-One — Kiwoom REST API (SMTM_AIO_kw)

**목표**: 키움 REST API를 통해 KOSPI/KOSDAQ 전체 종목 중
최근 10분간 1분봉 종가가 **지속 상승** 중인 종목을 상승비율 순으로 선별하여
자동 매매하는 멀티에셋 주식 전략의 전체 데이터 흐름을 보여준다.

## 암호화폐 → 주식 주요 변경사항

| 항목 | 암호화폐 (Upbit) | 주식 (Kiwoom) |
|------|-----------------|---------------|
| 인증 | 불필요 (공개 API) | APP_KEY + APP_SECRET → Bearer 토큰 |
| 종목코드 | KRW-XRP 등 | 6자리 숫자 (005930) |
| 주문 단위 | 소수점 허용 | **정수 주(株)** 필수 |
| 호가 단위 | 없음 | 가격대별 1/5/10/50/100/500/1,000원 |
| 수수료 | 매수·매도 0.05% | 매수·매도 **0.015%** |
| 거래세 | 없음 | **매도 시** 0.18% (KOSPI/KOSDAQ) |
| 장 운영 | 24시간 | **평일 09:00~15:30** |
| 가격 제한 | 없음 | 전일 종가 ±30% |
| 결제 | 즉시 | T+2 (시뮬에서 즉시 처리) |

## 전체 데이터 흐름

```
[Kiwoom /oauth2/token]
        ↓  Bearer Access Token
[전체 종목 목록 조회 (KOSPI + KOSDAQ)]
        ↓  가격 필터 MIN_PRICE ~ MAX_PRICE
[Kiwoom 1분봉 조회 (종목별 10개)]
        ↓  closing_price 리스트 (시간순)
[지속 상승 판별] all(p[i] > p[i-1])
        ↓  상승비율 rise_ratio = (p9 - p0) / p0
[상승비율 내림차순 정렬 → 상위 TOP_N]
        ↓  primary_candle 리스트 (rise_ratio, rank 포함)
[StrategyStockRising.update_trading_info()]
        ↓  랭킹 갱신, 현재가 캐시
[StrategyStockRising.get_request()]
        ↓  호가 단위 조정된 buy / sell 주문 (정수 주)
[MockStockTrader.send_request()]
        ↓  즉시 체결 → 수수료 + 거래세 계산 → callback
[StrategyStockRising.update_result()]
        ↓  잔고 / 보유량 반영
[결과 분석 및 시각화]
```

> **DEMO_MODE = True**: API 키 없이 합성 데이터로 실행 가능.
> 실거래 연동 시 `APP_KEY`, `APP_SECRET` 설정 후 `DEMO_MODE = False` 로 변경.


In [ ]:
%matplotlib inline
import sys
import os
import logging
import copy
import math
import time
import random
from datetime import datetime, timedelta

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display

# ── 프로젝트 루트를 sys.path 에 추가 ─────────────────────────────────────────
_ROOT = os.path.abspath(".")
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)

# smtm 모듈 (analyzer 제외 — matplotlib.use("Agg") 충돌 방지)
from smtm.date_converter import DateConverter
from smtm.strategy.strategy import Strategy

# ── Kiwoom REST API 설정 ──────────────────────────────────────────────────────
KIWOOM_BASE      = "https://api.kiwoom.com"
KIWOOM_TOKEN_URL = f"{KIWOOM_BASE}/oauth2/token"
KIWOOM_PRICE_URL = f"{KIWOOM_BASE}/api/dostk/stkinfo"      # 현재가 (OPT10001)
KIWOOM_CHART_URL = f"{KIWOOM_BASE}/api/dostk/minchart"     # 1분봉 (OPT10080)
KIWOOM_LIST_URL  = f"{KIWOOM_BASE}/api/dostk/mrktlstinfo"  # 시장 종목 조회

APP_KEY    = ""   # 키움 APP_KEY    (DEMO_MODE=False 시 필수)
APP_SECRET = ""   # 키움 APP_SECRET (DEMO_MODE=False 시 필수)

# ── 전역 파라미터 ─────────────────────────────────────────────────────────────
DEMO_MODE      = True      # True: 합성 데이터 / False: 실시간 Kiwoom API
MIN_PRICE      = 5_000     # 가격 하한 (KRW) — 너무 저가 종목 제외
MAX_PRICE      = 200_000   # 가격 상한 (KRW) — 초고가 종목 제외
CANDLE_COUNT   = 10        # 지속 상승 확인할 1분봉 수
TOP_N          = 15        # 상위 반환 종목 수
INITIAL_BUDGET = 5_000_000 # 초기 예산 (KRW) — 5백만원
MAX_BUY_AMOUNT = 1_000_000 # 종목당 최대 매수 금액 (KRW) — 1백만원
N_TURNS        = 5         # 시뮬레이션 턴 수

# ── 주식 거래 비용 ────────────────────────────────────────────────────────────
BUY_COMMISSION   = 0.00015  # 매수 수수료 0.015% (키움 온라인)
SELL_COMMISSION  = 0.00015  # 매도 수수료 0.015%
SELL_TAX_KOSPI   = 0.0018   # 거래세 0.18% (KOSPI, 2024년~)
SELL_TAX_KOSDAQ  = 0.0018   # 거래세 0.18% (KOSDAQ, 2024년~)

print(f"DEMO_MODE      : {DEMO_MODE}")
print(f"가격 필터       : {MIN_PRICE:,} ~ {MAX_PRICE:,} KRW")
print(f"지속 상승 확인   : {CANDLE_COUNT}분봉")
print(f"초기 예산        : {INITIAL_BUDGET:,} KRW")
print(f"종목당 최대 매수  : {MAX_BUY_AMOUNT:,} KRW")
print(f"Kiwoom Base URL : {KIWOOM_BASE}")


In [ ]:
class NbLogCapture(logging.Handler):
    """
    셀 실행 중 발생하는 로그를 내부에 누적.
    에러 발생 시 flush_on_error() 를 호출하면 전체 로그를 출력한다.
    """
    def __init__(self):
        super().__init__()
        self.records = []
        self.setFormatter(
            logging.Formatter("[%(levelname)s] %(name)s: %(message)s")
        )

    def emit(self, record):
        self.records.append(self.format(record))

    def flush_on_error(self, prefix="에러 발생 시 수집된 로그"):
        if self.records:
            print("\n" + "=" * 64)
            print(f" {prefix}")
            print("=" * 64)
            for r in self.records:
                print(r)
            print("=" * 64)
        self.records.clear()

    def clear(self):
        self.records.clear()


nb_log = NbLogCapture()
nb_log.setLevel(logging.DEBUG)

logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    force=True,
)
print("로깅 설정 완료")


## 섹션 1 — Kiwoom REST API 인증 + 전체 종목 조회 + 가격 필터

**데이터 흐름**

```
POST /oauth2/token  (APP_KEY + APP_SECRET)
        ↓  Bearer Access Token
GET /api/dostk/mrktlstinfo  (market=K0:KOSPI, K1:KOSDAQ)
        ↓  전체 종목 목록
GET /api/dostk/stkinfo  (배치 또는 개별)
        ↓  현재가
가격 필터 (MIN_PRICE ~ MAX_PRICE)
        ↓  price_filtered  {code: {name, market_type, price}}
```

**주요 응답 필드**
| 필드 | 설명 |
|------|------|
| `stk_code` | 종목코드 (6자리) |
| `stk_name` | 종목명 |
| `cur_prc`  | 현재가 (부호 포함, 절대값 사용) |
| `prdy_vrss_sign` | 2=상승, 5=하락, 3=보합 |


In [ ]:
# ── 데모용 KOSPI / KOSDAQ 대표 종목 ─────────────────────────────────────────
DEMO_STOCK_LIST = [
    # KOSPI — 가격 필터 통과 예상 종목
    {"code": "005930", "name": "삼성전자",       "market": "KOSPI",  "price": 67_400},
    {"code": "003550", "name": "LG",             "market": "KOSPI",  "price": 68_000},
    {"code": "066570", "name": "LG전자",         "market": "KOSPI",  "price": 73_600},
    {"code": "055550", "name": "신한지주",       "market": "KOSPI",  "price": 47_250},
    {"code": "105560", "name": "KB금융",         "market": "KOSPI",  "price": 68_400},
    {"code": "086790", "name": "하나금융지주",   "market": "KOSPI",  "price": 52_400},
    {"code": "035720", "name": "카카오",         "market": "KOSPI",  "price": 38_500},
    {"code": "028260", "name": "삼성물산",       "market": "KOSPI",  "price": 106_500},
    {"code": "068270", "name": "셀트리온",       "market": "KOSPI",  "price": 148_000},
    {"code": "352820", "name": "하이브",         "market": "KOSPI",  "price": 165_000},
    # KOSPI — 가격 필터 밖 (테스트용)
    {"code": "035420", "name": "NAVER",          "market": "KOSPI",  "price": 185_000},
    {"code": "000660", "name": "SK하이닉스",     "market": "KOSPI",  "price": 167_000},
    {"code": "207940", "name": "삼성바이오",     "market": "KOSPI",  "price": 740_000},
    # KOSDAQ — 가격 필터 통과 예상 종목
    {"code": "263750", "name": "펄어비스",       "market": "KOSDAQ", "price": 39_850},
    {"code": "251270", "name": "넷마블",         "market": "KOSDAQ", "price": 49_200},
    {"code": "112040", "name": "위메이드",       "market": "KOSDAQ", "price": 24_850},
    {"code": "041510", "name": "에스엠",         "market": "KOSDAQ", "price": 65_800},
    {"code": "035900", "name": "JYP엔터",        "market": "KOSDAQ", "price": 49_000},
    {"code": "086520", "name": "에코프로",       "market": "KOSDAQ", "price": 65_200},
    {"code": "196170", "name": "알테오젠",       "market": "KOSDAQ", "price": 148_000},
    {"code": "293490", "name": "카카오게임즈",   "market": "KOSDAQ", "price": 14_100},
]

# ── Kiwoom 인증 ──────────────────────────────────────────────────────────────
access_token = None  # 전역 토큰

def _get_access_token(app_key, app_secret):
    """
    키움 REST API 토큰 발급.

    POST {KIWOOM_TOKEN_URL}
    Body: grant_type=client_credentials&appkey={APP_KEY}&appsecret={APP_SECRET}
    Response: {token_type, access_token, expires_in}
    """
    resp = requests.post(
        KIWOOM_TOKEN_URL,
        data={
            "grant_type": "client_credentials",
            "appkey":     app_key,
            "appsecret":  app_secret,
        },
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        timeout=10,
    )
    resp.raise_for_status()
    return resp.json()["access_token"]


nb_log.clear()
try:
    if DEMO_MODE:
        access_token = "DEMO_TOKEN"
        all_stocks = DEMO_STOCK_LIST
        print(f"[DEMO] 샘플 종목 {len(all_stocks)}개 사용")
    else:
        print("Kiwoom 토큰 발급 중...")
        access_token = _get_access_token(APP_KEY, APP_SECRET)
        print(f"토큰 발급 완료 (앞 20자): {access_token[:20]}...")

        # 실제 API: 시장별 종목 리스트 조회
        headers = {
            "Authorization": f"Bearer {access_token}",
            "Content-Type":  "application/json",
            "apikey":        APP_KEY,
        }
        all_stocks = []
        for market_code, market_name in [("K0", "KOSPI"), ("K1", "KOSDAQ")]:
            resp = requests.get(
                KIWOOM_LIST_URL,
                params={"market": market_code},
                headers=headers,
                timeout=10,
            )
            resp.raise_for_status()
            for item in resp.json().get("output", []):
                all_stocks.append({
                    "code":   item["stk_code"],
                    "name":   item["stk_name"],
                    "market": market_name,
                    "price":  0,   # 현재가는 별도 조회
                })
        print(f"전체 종목 수: {len(all_stocks)}개")

    print(f"\n첫 5개 샘플:")
    for s in all_stocks[:5]:
        print(f"  {s['code']}  {s['name']:<12s}  {s['market']}")

except requests.exceptions.RequestException as e:
    nb_log.flush_on_error("Kiwoom API 인증/종목 조회 실패")
    raise
except Exception as e:
    nb_log.flush_on_error("종목 목록 처리 오류")
    raise


In [ ]:
# ── 현재가 조회 + 가격 필터 ──────────────────────────────────────────────────

def _fetch_price_real(code, headers):
    """
    Kiwoom 현재가 조회.

    GET {KIWOOM_PRICE_URL}?stk_code={code}
    Response: {"output": {"stk_code": "005930", "cur_prc": "-67400", ...}}
    cur_prc 는 하락 시 '-' 부호 포함 → abs() 처리 필요.
    """
    resp = requests.get(
        KIWOOM_PRICE_URL,
        params={"stk_code": code},
        headers=headers,
        timeout=5,
    )
    resp.raise_for_status()
    raw = resp.json().get("output", {}).get("cur_prc", "0")
    return abs(int(raw.replace(",", "")))


nb_log.clear()
try:
    if DEMO_MODE:
        # 데모: DEMO_STOCK_LIST 의 price 필드 그대로 사용
        for s in all_stocks:
            pass   # price 이미 설정됨
        print(f"[DEMO] 현재가 {len(all_stocks)}개 종목 사용")
    else:
        headers = {
            "Authorization": f"Bearer {access_token}",
            "Content-Type":  "application/json",
            "apikey":        APP_KEY,
        }
        print(f"현재가 조회 중 ({len(all_stocks)}개 종목)...")
        for i, s in enumerate(all_stocks):
            s["price"] = _fetch_price_real(s["code"], headers)
            time.sleep(0.1)   # Kiwoom API 레이트 리밋
            if (i + 1) % 50 == 0:
                print(f"  {i+1}/{len(all_stocks)} 완료...")
        print("현재가 조회 완료")

    # ── 가격 필터 적용 ─────────────────────────────────────────────────────
    # price_filtered: {code: {name, market, price}}
    price_filtered = {
        s["code"]: {"name": s["name"], "market": s["market"], "price": s["price"]}
        for s in all_stocks
        if MIN_PRICE <= s["price"] <= MAX_PRICE
    }

    print(f"\n전체 종목       : {len(all_stocks)}개")
    print(f"가격 필터 조건   : {MIN_PRICE:,} ~ {MAX_PRICE:,} KRW")
    print(f"가격 필터 후     : {len(price_filtered)}개")
    print(f"제외된 종목      : {len(all_stocks) - len(price_filtered)}개")

    df_price = pd.DataFrame([
        {"코드": c, "종목명": v["name"], "시장": v["market"],
         "현재가(KRW)": f"{v['price']:,}"}
        for c, v in sorted(price_filtered.items(), key=lambda x: -x[1]["price"])
    ])
    print("\n가격 내림차순 상위 12개:")
    display(df_price.head(12).to_string(index=False))

except requests.exceptions.RequestException as e:
    nb_log.flush_on_error("현재가 API 조회 실패")
    raise
except Exception as e:
    nb_log.flush_on_error("가격 필터 처리 오류")
    raise


## 섹션 2 — 10분간 지속 상승 필터 + 상승비율 랭킹

**판별 기준 (암호화폐와 동일)**

```
candles = [c0, c1, ..., c9]  # 1분봉 종가, 시간순
지속 상승 ↔ all(c[i] > c[i-1] for i in 1..9)
상승비율   = (c9 - c0) / c0
```

**주식 특이사항**
- 종가는 **호가 단위** 단위로 움직임 → 연속 상승 조건이 암호화폐보다 엄격
- 장 중에만 1분봉 데이터가 생성됨 (09:00~15:30)
- 데이터는 시간 내림차순(최신 먼저) 반환 → **reverse() 필수**

**Kiwoom 1분봉 응답 필드**
| 필드 | 설명 |
|------|------|
| `stk_bsop_date` | 영업일 (YYYYMMDD) |
| `stk_bsop_hour` | 시간 (HHMMSS) |
| `stk_cur_prc`   | 해당 분봉 종가 (부호 포함) |
| `acml_vol`      | 누적 거래량 |


In [ ]:
# ── 호가 단위 함수 ───────────────────────────────────────────────────────────

def calc_tick_size(price: float) -> int:
    """
    국내 주식시장 호가 단위 계산 (2023년 기준).

    가격대       호가 단위
    ─────────────────────
    ~1,000원       1원
    1,000~5,000    5원
    5,000~10,000  10원
    10,000~50,000 50원
    50,000~100,000 100원
    100,000~500,000 500원
    500,000~       1,000원
    """
    p = int(price)
    if p < 1_000:      return 1
    elif p < 5_000:    return 5
    elif p < 10_000:   return 10
    elif p < 50_000:   return 50
    elif p < 100_000:  return 100
    elif p < 500_000:  return 500
    else:              return 1_000


def floor_to_tick(price: float) -> int:
    """호가 단위 아래로 절사 (매수 주문 시 사용)."""
    tick = calc_tick_size(price)
    return (int(price) // tick) * tick


def ceil_to_tick(price: float) -> int:
    """호가 단위 위로 올림 (시장가 근사 주문 시 사용)."""
    tick = calc_tick_size(price)
    p = int(price)
    return p if p % tick == 0 else (p // tick + 1) * tick


# ── 장 운영시간 확인 ──────────────────────────────────────────────────────────

def is_market_open() -> bool:
    """
    평일 09:00~15:30 인지 확인 (한국 로컬 시간 기준).
    장 전 동시호가(08:00~09:00), 장 후 시간외(15:40~18:00)는 제외.
    """
    now = datetime.now()
    if now.weekday() >= 5:   # 토(5), 일(6)
        return False
    t_open  = now.replace(hour=9,  minute=0,  second=0, microsecond=0)
    t_close = now.replace(hour=15, minute=30, second=0, microsecond=0)
    return t_open <= now <= t_close


# 기능 검증
print("호가 단위 검증:")
for price in [500, 3000, 7500, 30000, 75000, 200000, 600000]:
    tick = calc_tick_size(price)
    print(f"  {price:>8,}원  →  호가 단위: {tick:>5,}원")

print(f"\n현재 장 운영시간 여부: {is_market_open()}")
if not is_market_open():
    print("  (장 외 시간 — DEMO_MODE 에서는 무관하게 실행 가능)")


In [ ]:
# ── 1분봉 조회 헬퍼 ──────────────────────────────────────────────────────────

def _fetch_real_candles(code, headers, count=10):
    """
    Kiwoom 1분봉 종가 리스트 반환 (시간순, 오래된 것부터).

    GET {KIWOOM_CHART_URL}?stk_code={code}&tic_scope=1&qry_count={count}
    Response: {"output": [{stk_bsop_date, stk_bsop_hour, stk_cur_prc, ...}, ...]}
    키움은 최신 데이터를 먼저 반환 → reverse() 필수.
    """
    resp = requests.get(
        KIWOOM_CHART_URL,
        params={"stk_code": code, "tic_scope": "1", "qry_count": count},
        headers=headers,
        timeout=5,
    )
    resp.raise_for_status()
    data = resp.json().get("output", [])
    data.reverse()   # 최신순 → 시간순
    return [abs(int(item["stk_cur_prc"].replace(",", ""))) for item in data]


def _make_demo_candles(base_price: int, count: int = 10) -> list:
    """
    합성 1분봉 종가 리스트 생성 (호가 단위 적용).
    주식은 호가 단위 단위로만 움직이므로 단순 소수점 가격은 부적절.
    """
    tick = calc_tick_size(base_price)
    prices = [base_price]
    for _ in range(count - 1):
        # 호가 단위의 ±3배 범위 내에서 랜덤 이동
        delta_ticks = random.randint(-3, 5)
        new_price = prices[-1] + delta_ticks * tick
        # 음수 방지
        new_price = max(tick, new_price)
        prices.append(new_price)
    return prices


def is_continuously_rising(prices: list) -> bool:
    """종가 리스트가 엄격 단조 증가인지 확인."""
    if len(prices) < 2:
        return False
    return all(prices[i] > prices[i - 1] for i in range(1, len(prices)))


def calc_rise_ratio(prices: list) -> float:
    """상승비율 = (마지막 종가 - 첫 종가) / 첫 종가."""
    if not prices or prices[0] == 0:
        return 0.0
    return (prices[-1] - prices[0]) / prices[0]


# 검증
sample = [50000, 50050, 50100, 50200, 50300, 50400, 50500, 50600, 50700, 50800]
print(f"is_continuously_rising (정상 단조증가): {is_continuously_rising(sample)}")  # True
broken = sample[:]; broken[5] = 50100  # 50400 → 50100: 이전(50300) 보다 낮음
print(f"is_continuously_rising (하락 포함):    {is_continuously_rising(broken)}")  # False
equal  = sample[:]; equal[5]  = equal[4]  # 50400 == 50300: strict 조건 미충족
print(f"is_continuously_rising (동일가 포함):  {is_continuously_rising(equal)}")   # False
print(f"calc_rise_ratio: {calc_rise_ratio(sample)*100:.4f}%")


In [ ]:
nb_log.clear()

if not DEMO_MODE and not is_market_open():
    print("[경고] 현재 장 외 시간입니다. 실시간 데이터가 없어 분봉 조회 결과가 비어있을 수 있습니다.")
    print("         DEMO_MODE = True 로 설정하거나 장 중에 실행하세요.")

print(f"총 {len(price_filtered)}개 종목 대상으로 10분 지속 상승 탐색 중...")

rising_assets = []
headers_live = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type":  "application/json",
    "apikey":        APP_KEY,
}

try:
    for idx, (code, info) in enumerate(price_filtered.items()):
        if DEMO_MODE:
            candles = _make_demo_candles(info["price"])
        else:
            candles = _fetch_real_candles(code, headers_live, CANDLE_COUNT)
            time.sleep(0.12)   # Kiwoom 레이트 리밋

        if not candles:
            continue

        if is_continuously_rising(candles):
            ratio = calc_rise_ratio(candles)
            rising_assets.append({
                "code":          code,
                "name":          info["name"],
                "market":        info["market"],
                "first_price":   candles[0],
                "current_price": candles[-1],
                "rise_ratio":    ratio,
                "candles":       candles,
            })

        if (idx + 1) % 5 == 0:
            print(f"  {idx+1}/{len(price_filtered)} 처리 완료...")

    rising_assets.sort(key=lambda x: x["rise_ratio"], reverse=True)
    for rank, a in enumerate(rising_assets, start=1):
        a["rank"] = rank

    print(f"\n탐색 완료")
    print(f"  지속 상승 종목 : {len(rising_assets)}개 / {len(price_filtered)}개")

    if rising_assets:
        df_r = pd.DataFrame([{
            "순위":      a["rank"],
            "코드":      a["code"],
            "종목명":    a["name"],
            "시장":      a["market"],
            "첫 종가":   f"{a['first_price']:,}",
            "현재 종가": f"{a['current_price']:,}",
            "상승비율":  f"{a['rise_ratio']*100:.4f}%",
        } for a in rising_assets])
        display(df_r.to_string(index=False))
    else:
        print("  현재 조건을 만족하는 종목이 없습니다.")
        print("  Tip: 호가 단위 때문에 주식은 crypto보다 지속 상승 조건이 엄격합니다.")
        print("       CANDLE_COUNT 를 줄이거나 다시 실행하세요.")

except requests.exceptions.RequestException as e:
    nb_log.flush_on_error("분봉 조회 API 실패")
    raise
except Exception as e:
    nb_log.flush_on_error("지속 상승 탐색 오류")
    raise


In [ ]:
if rising_assets:
    top_n_viz = min(TOP_N, len(rising_assets))
    top = rising_assets[:top_n_viz]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(
        "10분간 지속 상승 종목 — KOSPI/KOSDAQ 필터링 결과",
        fontsize=13, fontweight="bold"
    )

    # 왼쪽: 상승비율 수평 막대 (종목명 표시)
    ax1 = axes[0]
    labels  = [f"{a['name']}({a['code']})" for a in reversed(top)]
    ratios  = [a["rise_ratio"] * 100 for a in reversed(top)]
    max_r   = max(ratios) if ratios else 1
    colors  = plt.cm.RdYlGn([r / max_r for r in ratios])
    bars = ax1.barh(labels, ratios, color=colors, edgecolor="white")
    ax1.set_xlabel("상승비율 (%)")
    ax1.set_title(f"상위 {top_n_viz}종목 상승비율 랭킹")
    ax1.grid(axis="x", alpha=0.3)
    for bar, ratio in zip(bars, ratios):
        ax1.text(
            bar.get_width() + max_r * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{ratio:.4f}%", va="center", fontsize=8
        )

    # 오른쪽: 상위 5종목 가격 추이 (절대가 → 변화율%)
    ax2 = axes[1]
    for a in rising_assets[:5]:
        base = a["candles"][0]
        norm = [(p - base) / base * 100 for p in a["candles"]]
        ax2.plot(range(len(norm)), norm, marker="o", markersize=3,
                 label=f"{a['name']}")
    ax2.set_xlabel("분봉 인덱스 (0=10분 전, 9=현재)")
    ax2.set_ylabel("변화율 (%)")
    ax2.set_title("상위 5종목 10분간 종가 변화율")
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.3)
    ax2.axhline(0, color="black", linestyle="--", linewidth=0.5)

    plt.tight_layout()
    os.makedirs("output", exist_ok=True)
    plt.savefig("output/kw_s2_rising_rank.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("차트 저장: output/kw_s2_rising_rank.png")
else:
    print("시각화할 데이터 없음")


## 섹션 3 — `KosdaqRisingDataProvider` 구현

섹션 1~2의 로직을 `get_info()` 한 번의 호출로 캡슐화.

**반환 스키마** (`primary_candle` + 주식 확장 필드)

```python
{
    "type":          "primary_candle",
    "market":        "005930",          # 종목코드 (strategy/trader 라우팅 키)
    "name":          "삼성전자",          # 종목명
    "market_type":   "KOSPI",           # 시장 구분
    "date_time":     "2025-03-15T10:05:00",
    "opening_price": 67_000,            # 10분 전 첫 종가 (호가 단위 정수)
    "high_price":    67_500,            # 10분간 최고가
    "low_price":     66_800,            # 10분간 최저가
    "closing_price": 67_400,            # 현재 종가
    "acc_price":     0,
    "acc_volume":    0,
    "rise_ratio":    0.006,             # 상승비율 (소수)
    "rank":          1,                 # 상승비율 순위
}
```


In [ ]:
class KosdaqRisingDataProvider:
    """
    Kiwoom REST API 를 통해 KOSPI/KOSDAQ 전체 종목 중
    최근 CANDLE_COUNT 분봉 동안 종가가 지속 상승한 종목을
    상승비율 순으로 반환하는 데이터 프로바이더.

    데이터 흐름:
        [Kiwoom 인증] → Bearer token
        [전체 종목 목록] → 가격 필터 적용
        [1분봉 조회 (종목별)] → is_continuously_rising() 판별
        [상승비율 내림차순 정렬] → 상위 TOP_N → primary_candle 리스트 반환

    주식 특이사항:
        - 장 외 시간에는 candle 이 빈 리스트일 수 있음 → 빈 리스트 반환
        - 호가 단위 적용으로 1틱 변화만 있어도 상승으로 인정
        - 종가 부호(음수=하락): abs() 처리 필수
    """

    CODE = "KOSRISING"
    NAME = "KosdaqRising DataProvider"

    TOKEN_URL  = KIWOOM_TOKEN_URL
    PRICE_URL  = KIWOOM_PRICE_URL
    CHART_URL  = KIWOOM_CHART_URL
    LIST_URL   = KIWOOM_LIST_URL

    def __init__(
        self,
        app_key:       str   = "",
        app_secret:    str   = "",
        access_token:  str   = None,
        min_price:     int   = 5_000,
        max_price:     int   = 200_000,
        candle_count:  int   = 10,
        top_n:         int   = 20,
        demo_mode:     bool  = False,
        demo_stocks:   list  = None,    # DEMO_STOCK_LIST 형식
    ):
        self.app_key      = app_key
        self.app_secret   = app_secret
        self._token       = access_token
        self.min_price    = min_price
        self.max_price    = max_price
        self.candle_count = candle_count
        self.top_n        = top_n
        self.demo_mode    = demo_mode
        self.demo_stocks  = demo_stocks or []

        self.logger = logging.getLogger(self.__class__.__name__)
        self.logger.addHandler(nb_log)
        self.logger.setLevel(logging.DEBUG)

    # ── 내부 헬퍼 ─────────────────────────────────────────────────────────

    def _get_token(self) -> str:
        if self._token:
            return self._token
        resp = requests.post(
            self.TOKEN_URL,
            data={"grant_type": "client_credentials",
                  "appkey": self.app_key, "appsecret": self.app_secret},
            headers={"Content-Type": "application/x-www-form-urlencoded"},
            timeout=10,
        )
        resp.raise_for_status()
        self._token = resp.json()["access_token"]
        return self._token

    def _headers(self) -> dict:
        return {
            "Authorization": f"Bearer {self._get_token()}",
            "Content-Type":  "application/json",
            "apikey":        self.app_key,
        }

    def _all_stocks(self) -> list:
        if self.demo_mode:
            return self.demo_stocks
        stocks = []
        h = self._headers()
        for mcode, mname in [("K0", "KOSPI"), ("K1", "KOSDAQ")]:
            r = requests.get(self.LIST_URL, params={"market": mcode},
                             headers=h, timeout=10)
            r.raise_for_status()
            for item in r.json().get("output", []):
                stocks.append({"code": item["stk_code"],
                                "name": item["stk_name"],
                                "market": mname, "price": 0})
        return stocks

    def _get_price(self, code: str, h: dict) -> int:
        r = requests.get(self.PRICE_URL,
                         params={"stk_code": code}, headers=h, timeout=5)
        r.raise_for_status()
        raw = r.json().get("output", {}).get("cur_prc", "0")
        return abs(int(raw.replace(",", "")))

    def _get_candles(self, code: str, base_price: int, h: dict) -> list:
        if self.demo_mode and base_price:
            tick = calc_tick_size(base_price)
            prices = [base_price]
            for _ in range(self.candle_count - 1):
                d = random.randint(-3, 5)
                prices.append(max(tick, prices[-1] + d * tick))
            return prices
        try:
            r = requests.get(
                self.CHART_URL,
                params={"stk_code": code,
                        "tic_scope": "1",
                        "qry_count": self.candle_count},
                headers=h, timeout=5,
            )
            r.raise_for_status()
            data = r.json().get("output", [])
            data.reverse()
            return [abs(int(it["stk_cur_prc"].replace(",", ""))) for it in data]
        except Exception as e:
            self.logger.error(f"분봉 조회 실패 ({code}): {e}")
            return []

    @staticmethod
    def _is_rising(prices):
        return len(prices) >= 2 and all(
            prices[i] > prices[i - 1] for i in range(1, len(prices))
        )

    @staticmethod
    def _rise_ratio(prices):
        return 0.0 if not prices or prices[0] == 0 else (prices[-1] - prices[0]) / prices[0]

    # ── 공개 인터페이스 ────────────────────────────────────────────────────

    def get_info(self) -> list:
        """
        KOSPI/KOSDAQ 전 종목 탐색 → 가격 필터 → 지속 상승 판별 →
        상승비율 순 정렬 → 상위 TOP_N primary_candle 리스트 반환.
        """
        self.logger.info("[get_info] 시작")

        stocks = self._all_stocks()
        self.logger.info(f"  전체 종목: {len(stocks)}개")

        # 가격 조회 + 필터
        h = self._headers()
        filtered = []
        for s in stocks:
            if not self.demo_mode:
                s["price"] = self._get_price(s["code"], h)
                time.sleep(0.1)
            if self.min_price <= s["price"] <= self.max_price:
                filtered.append(s)
        self.logger.info(f"  가격 필터 후: {len(filtered)}개")

        # 분봉 조회 + 지속 상승 판별
        rising = []
        for s in filtered:
            candles = self._get_candles(s["code"], s["price"], h)
            if not candles:
                continue
            if self._is_rising(candles):
                rising.append({**s, "candles": candles,
                               "ratio": self._rise_ratio(candles)})
            if not self.demo_mode:
                time.sleep(0.12)

        rising.sort(key=lambda x: x["ratio"], reverse=True)
        top = rising[: self.top_n]
        self.logger.info(f"  지속 상승: {len(rising)}개 → 상위 {len(top)}개 반환")

        now = datetime.now().strftime("%Y-%m-%dT%H:%M:%S")
        result = []
        for rank, item in enumerate(top, start=1):
            c = item["candles"]
            result.append({
                "type":          "primary_candle",
                "market":        item["code"],         # 종목코드가 라우팅 키
                "name":          item["name"],
                "market_type":   item["market"],       # KOSPI / KOSDAQ
                "date_time":     now,
                "opening_price": c[0],
                "high_price":    max(c),
                "low_price":     min(c),
                "closing_price": c[-1],
                "acc_price":     0,
                "acc_volume":    0,
                "rise_ratio":    item["ratio"],
                "rank":          rank,
            })
        return result


In [ ]:
nb_log.clear()
try:
    dp = KosdaqRisingDataProvider(
        app_key      = APP_KEY,
        app_secret   = APP_SECRET,
        access_token = access_token,
        min_price    = MIN_PRICE,
        max_price    = MAX_PRICE,
        candle_count = CANDLE_COUNT,
        top_n        = TOP_N,
        demo_mode    = DEMO_MODE,
        demo_stocks  = DEMO_STOCK_LIST if DEMO_MODE else None,
    )

    print("KosdaqRisingDataProvider.get_info() 호출...")
    t0 = time.time()
    candle_info = dp.get_info()
    elapsed = time.time() - t0

    print(f"\n반환 캔들 수: {len(candle_info)}개  (소요: {elapsed:.2f}초)")

    if candle_info:
        print("\n첫 번째 캔들 딕셔너리 (스키마 확인):")
        for k, v in candle_info[0].items():
            print(f"  {k:<20s}: {v}")

        print("\n상위 10종목 요약:")
        for item in candle_info[:10]:
            tick = calc_tick_size(item["closing_price"])
            print(
                f"  #{item['rank']:2d}  [{item['market_type']}]  "
                f"{item['name']:<10s}({item['market']})  "
                f"종가: {item['closing_price']:>9,}원  "
                f"호가단위: {tick:>5,}원  "
                f"상승비율: {item['rise_ratio']*100:.4f}%"
            )
    else:
        print("반환된 종목 없음")

except Exception as e:
    nb_log.flush_on_error("데이터 프로바이더 테스트 실패")
    raise


## 섹션 4 — `StrategyStockRising` 구현

**암호화폐 전략과의 핵심 차이**

| 항목 | 암호화폐 (StrategyKRWRising) | 주식 (StrategyStockRising) |
|------|------------------------------|---------------------------|
| 주문 수량 | 소수점 허용 | **정수 주(株) 필수** |
| 주문 가격 | 현재가 그대로 | **호가 단위 절사** (floor_to_tick) |
| 매도 비용 | 수수료만 | 수수료 + **거래세** |
| 잔고 변경 | balance 직접 | balance (KRW) 직접 |
| 보유량 | 소수점 | **정수 주** |

**매수 요청 생성 로직**

```python
order_price = floor_to_tick(current_price)   # 호가 단위 적사
shares      = int(budget / order_price)      # 정수 주 계산
shares      = max(1, shares)                 # 최소 1주
if order_price * shares > balance: shares -= 1
if shares <= 0: return None
```


In [ ]:
class StrategyStockRising(Strategy):
    """
    KOSPI/KOSDAQ 전 종목 중 10분간 지속 상승 종목에 투자하는 멀티에셋 주식 전략.

    주식 특화 사항:
    - 주문 수량: 정수 주 (shares)
    - 주문 가격: 호가 단위 절사 (floor_to_tick)
    - 매도 비용: 수수료(0.015%) + 거래세(0.18%)
    - 잔고 단위: KRW 정수
    """

    NAME = "Stock Rising Multi"
    CODE = "SRM"

    MAX_BUY_COUNT   = 3           # 동시 보유 최대 종목 수
    MAX_BUY_AMOUNT  = 1_000_000  # 종목당 최대 매수 금액 (KRW)
    STOP_LOSS_RATIO = 0.05       # 손절 기준 5% (주식은 변동폭이 더 큼)

    BUY_COST_RATIO  = BUY_COMMISSION                   # 매수 총 비용률
    SELL_COST_RATIO = SELL_COMMISSION + SELL_TAX_KOSPI  # 매도 총 비용률 (KOSPI 기준)

    def __init__(self):
        self.is_initialized   = False
        self.budget           = 0
        self.balance          = 0
        self.min_price        = 0

        # {code: {"shares": int, "avg_price": int, "current_price": int,
        #         "market_type": str, "name": str}}
        self.holdings         = {}

        # [(code, rise_ratio), ...]  현재 턴 랭킹
        self.rankings         = []

        # {code: {"price": int, "name": str, "market_type": str}}
        self._last_info       = {}

        # {req_id: {"market": code, "type": str}}
        self.waiting_requests = {}

        self.result           = []
        self.logger = logging.getLogger(self.__class__.__name__)
        self.logger.addHandler(nb_log)
        self.logger.setLevel(logging.DEBUG)

    # ── Strategy 추상 메서드 구현 ──────────────────────────────────────────

    def initialize(self, budget, min_price=5_000,
                   add_spot_callback=None,
                   add_line_callback=None,
                   alert_callback=None):
        if self.is_initialized:
            return
        self.budget      = budget
        self.balance     = budget
        self.min_price   = min_price
        self.is_initialized = True
        self.logger.info(
            f"[{self.NAME}] 초기화: budget={budget:,}, "
            f"max_buy={self.MAX_BUY_COUNT}, "
            f"stop_loss={self.STOP_LOSS_RATIO*100:.1f}%"
        )

    def update_trading_info(self, info):
        """
        KosdaqRisingDataProvider.get_info() 반환값으로
        랭킹 / 현재가 / 종목명 캐시를 갱신한다.
        """
        if not self.is_initialized:
            return

        new_rankings = []
        for item in info:
            if not isinstance(item, dict) or item.get("type") != "primary_candle":
                continue

            code  = item["market"]           # 종목코드
            price = int(item.get("closing_price", 0))
            ratio = float(item.get("rise_ratio", 0))
            name  = item.get("name", code)
            mtype = item.get("market_type", "")

            self._last_info[code] = {"price": price, "name": name,
                                     "market_type": mtype}

            if code in self.holdings:
                self.holdings[code]["current_price"] = price

            new_rankings.append((code, ratio))

        new_rankings.sort(key=lambda x: x[1], reverse=True)
        self.rankings = new_rankings

        self.logger.info(
            "[update_trading_info] 랭킹 갱신: "
            + str([(c, f"{r*100:.4f}%") for c, r in self.rankings[:5]])
        )

    def get_request(self):
        """
        매도 우선: 손절 / 추세 소멸 → 매수: 상위 종목 정수 주 주문 생성.

        Returns: 요청 딕셔너리 리스트 or None
        """
        if not self.is_initialized or not self.rankings:
            return None

        requests_list = []
        ranked_set    = {c for c, _ in self.rankings}
        pending_sell  = {v["market"] for v in self.waiting_requests.values()
                         if v["type"] == "sell"}
        pending_buy   = {v["market"] for v in self.waiting_requests.values()
                         if v["type"] == "buy"}

        # ── 1. 매도 체크 ──────────────────────────────────────────────────
        for code in list(self.holdings.keys()):
            if code in pending_sell:
                continue
            holding = self.holdings[code]
            cur     = holding.get("current_price", holding["avg_price"])
            stop    = holding["avg_price"] * (1 - self.STOP_LOSS_RATIO)

            if cur < stop:
                req = self._create_sell_req(code, cur)
                if req:
                    requests_list.append(req)
                    self.logger.info(
                        f"[SELL/stop_loss] {holding.get('name', code)}({code}) "
                        f"@ {cur:,} (avg={holding['avg_price']:,})"
                    )
                continue

            if code not in ranked_set:
                req = self._create_sell_req(code, cur)
                if req:
                    requests_list.append(req)
                    self.logger.info(
                        f"[SELL/trend_over] {holding.get('name', code)}({code}) "
                        f"@ {cur:,}"
                    )

        # ── 2. 매수 체크 ──────────────────────────────────────────────────
        active_count = (
            len([c for c in self.holdings if c not in pending_sell])
            + len(pending_buy)
        )

        for code, ratio in self.rankings:
            if active_count >= self.MAX_BUY_COUNT:
                break
            if code in self.holdings or code in pending_buy:
                continue

            info  = self._last_info.get(code, {})
            price = info.get("price", 0)
            if price <= 0:
                continue

            buy_budget = min(self.MAX_BUY_AMOUNT, self.balance)
            if buy_budget < self.min_price:
                self.logger.warning(f"잔고 부족으로 매수 중단 (balance={self.balance:,})")
                break

            req = self._create_buy_req(code, price, buy_budget)
            if req:
                requests_list.append(req)
                active_count += 1
                rank_pos = [c for c, _ in self.rankings].index(code) + 1
                self.logger.info(
                    f"[BUY rank={rank_pos}, rise={ratio*100:.4f}%] "
                    f"{info.get('name', code)}({code}) @ {price:,}  "
                    f"{req['amount']}주"
                )

        return requests_list if requests_list else None

    def update_result(self, result):
        """체결 결과 반영 — 주식은 정수 주 단위로 보유량 변경."""
        if not isinstance(result, dict) or result.get("state") != "done":
            return

        req        = result.get("request", {})
        req_id     = req.get("id")
        if req_id in self.waiting_requests:
            del self.waiting_requests[req_id]

        code       = req.get("market")
        trade_type = result.get("type")
        price      = int(result.get("price", 0))
        shares     = int(result.get("amount", 0))
        msg        = result.get("msg", "")

        if not code or price <= 0 or shares <= 0 or msg != "success":
            return

        total  = price * shares
        mtype  = self._last_info.get(code, {}).get("market_type", "KOSPI")

        if trade_type == "buy":
            fee = round(total * self.BUY_COST_RATIO)
            self.balance -= (total + fee)
            if code not in self.holdings:
                info = self._last_info.get(code, {})
                self.holdings[code] = {
                    "shares":        0,
                    "avg_price":     0,
                    "current_price": price,
                    "name":          info.get("name", code),
                    "market_type":   info.get("market_type", "KOSPI"),
                }
            old    = self.holdings[code]
            new_sh = old["shares"] + shares
            self.holdings[code]["avg_price"] = round(
                (old["avg_price"] * old["shares"] + total) / new_sh
            )
            self.holdings[code]["shares"] = new_sh
            self.logger.info(
                f"[BUY done] {old.get('name',code)}({code}) "
                f"{shares}주 @ {price:,}  fee={fee:,}  balance={self.balance:,}"
            )

        elif trade_type == "sell":
            tax_rate = SELL_TAX_KOSDAQ if mtype == "KOSDAQ" else SELL_TAX_KOSPI
            cost = round(total * (SELL_COMMISSION + tax_rate))
            self.balance += (total - cost)
            if code in self.holdings:
                new_sh = self.holdings[code]["shares"] - shares
                if new_sh <= 0:
                    del self.holdings[code]
                else:
                    self.holdings[code]["shares"] = new_sh
            self.logger.info(
                f"[SELL done] {code} {shares}주 @ {price:,}  "
                f"cost(fee+tax)={cost:,}  balance={self.balance:,}"
            )

        self.result.append(copy.deepcopy(result))

    # ── Private 헬퍼 ──────────────────────────────────────────────────────

    def _create_buy_req(self, code: str, price: int, budget: int):
        """
        정수 주 매수 요청 생성.
        주문 가격은 호가 단위 아래로 절사 (보수적 주문).
        """
        order_price = floor_to_tick(price)
        if order_price <= 0:
            return None

        # 수수료 고려 후 최대 구매 가능 주수
        net_budget = budget / (1 + self.BUY_COST_RATIO)
        shares     = int(net_budget // order_price)
        shares     = max(1, shares)

        # 실제 비용 재확인
        total_cost = order_price * shares * (1 + self.BUY_COST_RATIO)
        while shares > 0 and total_cost > self.balance:
            shares    -= 1
            total_cost = order_price * shares * (1 + self.BUY_COST_RATIO)

        if shares <= 0:
            return None

        req_id = DateConverter.timestamp_id()
        self.waiting_requests[req_id] = {"market": code, "type": "buy"}
        info = self._last_info.get(code, {})
        return {
            "id":          req_id,
            "type":        "buy",
            "market":      code,
            "name":        info.get("name", code),
            "market_type": info.get("market_type", "KOSPI"),
            "price":       order_price,
            "amount":      shares,
            "date_time":   datetime.now().strftime("%Y-%m-%dT%H:%M:%S"),
        }

    def _create_sell_req(self, code: str, price: int):
        """전량 매도 요청 생성. 주문 가격은 호가 단위 내림 적용."""
        holding = self.holdings.get(code)
        if not holding or holding["shares"] <= 0:
            return None

        order_price = floor_to_tick(price)
        req_id = DateConverter.timestamp_id()
        self.waiting_requests[req_id] = {"market": code, "type": "sell"}
        return {
            "id":          req_id,
            "type":        "sell",
            "market":      code,
            "name":        holding.get("name", code),
            "market_type": holding.get("market_type", "KOSPI"),
            "price":       order_price,
            "amount":      holding["shares"],
            "date_time":   datetime.now().strftime("%Y-%m-%dT%H:%M:%S"),
        }


In [ ]:
nb_log.clear()
try:
    strategy = StrategyStockRising()
    strategy.initialize(INITIAL_BUDGET, min_price=MIN_PRICE)

    print("StrategyStockRising 초기화 완료")
    print(f"  이름           : {strategy.NAME}  (CODE={strategy.CODE})")
    print(f"  초기 예산       : {strategy.budget:,} KRW")
    print(f"  최대 보유 종목  : {strategy.MAX_BUY_COUNT}개")
    print(f"  종목당 최대 매수 : {strategy.MAX_BUY_AMOUNT:,} KRW")
    print(f"  손절 기준       : {strategy.STOP_LOSS_RATIO*100:.1f}%")
    print(f"  매수 비용률     : {strategy.BUY_COST_RATIO*100:.3f}% (수수료)")
    print(f"  매도 비용률     : {strategy.SELL_COST_RATIO*100:.3f}% (수수료+거래세)")

    # 호가 단위 적용 검증
    test_price = 67_450
    order_p    = floor_to_tick(test_price)
    tick       = calc_tick_size(test_price)
    print(f"\n  [호가 단위 검증] 현재가={test_price:,}원 → "
          f"호가단위={tick:,}원 → 주문가={order_p:,}원")

except Exception as e:
    nb_log.flush_on_error("전략 초기화 실패")
    raise


## 섹션 5 — `MockStockTrader` 구현

**암호화폐 `MockMultiTrader` 와의 차이**

| 항목 | 암호화폐 | 주식 |
|------|---------|------|
| 수량 타입 | float | **int (주)** |
| 매수 비용 | 수수료 0.05% | 수수료 **0.015%** |
| 매도 비용 | 수수료 0.05% | 수수료 0.015% + **거래세 0.18%** |
| 거래세 계산 | 없음 | 시장구분(KOSPI/KOSDAQ)별 적용 |

**매도 비용 계산식**

```
total      = price × shares
fee        = total × SELL_COMMISSION   # 0.015%
tax        = total × SELL_TAX_KOSPI    # 0.18%  (KOSDAQ 동일)
net_income = total - fee - tax
```


In [ ]:
class MockStockTrader:
    """
    주식 멀티에셋 모의 트레이더 (노트북 전용).

    실제 API 호출 없이 지정 가격으로 즉시 체결 처리.
    수수료(매수·매도 공통) + 거래세(매도 전용) 를 정확히 계산한다.
    실거래에서는 UpbitMultiTrader → KiwoomMultiTrader 로 교체한다.
    """

    NAME = "MockStockTrader"

    def __init__(self, initial_budget: int):
        self.balance   = initial_budget
        self.assets    = {}     # {code: shares (int)}
        self.trade_log = []
        self.logger    = logging.getLogger(self.__class__.__name__)
        self.logger.addHandler(nb_log)
        self.logger.setLevel(logging.DEBUG)

    def send_request(self, request_list, callback):
        for req in (request_list or []):
            result = self._fill_order(req)
            callback(result)

    def _fill_order(self, req):
        code       = req.get("market", "UNKNOWN")
        trade_type = req.get("type", "")
        price      = int(req.get("price", 0))
        shares     = int(req.get("amount", 0))
        mtype      = req.get("market_type", "KOSPI")
        now        = datetime.now().strftime("%Y-%m-%dT%H:%M:%S")

        base = {
            "request":     req,
            "type":        trade_type,
            "market":      code,
            "name":        req.get("name", code),
            "market_type": mtype,
            "price":       price,
            "amount":      shares,
            "state":       "done",
            "date_time":   now,
        }

        if trade_type == "cancel":
            return {**base, "msg": "success"}

        if price <= 0 or shares <= 0:
            return {**base, "msg": "invalid_order"}

        total = price * shares

        if trade_type == "buy":
            fee  = round(total * BUY_COMMISSION)
            cost = total + fee
            if cost > self.balance:
                self.logger.warning(
                    f"[BUY FAIL] {code} 잔고 부족 "
                    f"(필요={cost:,} / 보유={self.balance:,})"
                )
                return {**base, "msg": "insufficient_balance"}
            self.balance -= cost
            self.assets[code] = self.assets.get(code, 0) + shares

            self.trade_log.append({
                "type": "buy", "code": code, "name": req.get("name", code),
                "market": mtype, "price": price, "shares": shares,
                "total": total, "fee": fee, "tax": 0, "time": now,
            })
            self.logger.info(
                f"[BUY] {req.get('name',code)}({code}) {shares}주 @ {price:,}  "
                f"fee={fee:,}  balance={self.balance:,}"
            )
            return {**base, "msg": "success"}

        elif trade_type == "sell":
            have = self.assets.get(code, 0)
            if have < shares:
                self.logger.warning(
                    f"[SELL FAIL] {code} 보유량 부족 ({have}주 < {shares}주)"
                )
                return {**base, "msg": "insufficient_asset"}

            fee     = round(total * SELL_COMMISSION)
            tax_rate = SELL_TAX_KOSDAQ if mtype == "KOSDAQ" else SELL_TAX_KOSPI
            tax     = round(total * tax_rate)
            net     = total - fee - tax
            self.balance += net
            self.assets[code] = have - shares
            if self.assets[code] <= 0:
                del self.assets[code]

            self.trade_log.append({
                "type": "sell", "code": code, "name": req.get("name", code),
                "market": mtype, "price": price, "shares": shares,
                "total": total, "fee": fee, "tax": tax, "time": now,
            })
            self.logger.info(
                f"[SELL] {req.get('name',code)}({code}) {shares}주 @ {price:,}  "
                f"fee={fee:,}  tax={tax:,}  net={net:,}  balance={self.balance:,}"
            )
            return {**base, "msg": "success"}

        return {**base, "msg": "unknown_type"}

    def cancel_request(self, request_id):
        pass

    def cancel_all_requests(self):
        pass

    def get_account_info(self):
        return {
            "balance":   self.balance,
            "asset":     {k: v for k, v in self.assets.items() if v > 0},
            "date_time": datetime.now().strftime("%Y-%m-%dT%H:%M:%S"),
        }


In [ ]:
nb_log.clear()
try:
    test_trader = MockStockTrader(initial_budget=5_000_000)

    # ── 매수 테스트: 삼성전자 10주 ───────────────────────────────────────
    buy_req = {
        "id":          DateConverter.timestamp_id(),
        "type":        "buy",
        "market":      "005930",
        "name":        "삼성전자",
        "market_type": "KOSPI",
        "price":       67_400,        # 호가 단위 적용된 가격
        "amount":      10,
        "date_time":   datetime.now().strftime("%Y-%m-%dT%H:%M:%S"),
    }
    results = []
    test_trader.send_request([buy_req], lambda r: results.append(r))
    r = results[0]

    expected_fee  = 67_400 * 10 * BUY_COMMISSION
    print("매수 요청 결과 (삼성전자 10주 @ 67,400원):")
    for k in ("type", "market", "name", "price", "amount", "msg", "state"):
        print(f"  {k:<12}: {r[k]}")
    print(f"  예상 수수료  : {expected_fee:,.0f}원")
    print(f"  잔고         : {test_trader.balance:,} KRW")
    print(f"  보유         : {test_trader.assets}")

    # ── 매도 테스트: 삼성전자 10주 (KOSPI 거래세 포함) ──────────────────
    sell_req = {
        "id":          DateConverter.timestamp_id(),
        "type":        "sell",
        "market":      "005930",
        "name":        "삼성전자",
        "market_type": "KOSPI",
        "price":       67_900,
        "amount":      10,
        "date_time":   datetime.now().strftime("%Y-%m-%dT%H:%M:%S"),
    }
    results2 = []
    test_trader.send_request([sell_req], lambda r: results2.append(r))
    r2 = results2[0]

    sell_total = 67_900 * 10
    sell_fee   = round(sell_total * SELL_COMMISSION)
    sell_tax   = round(sell_total * SELL_TAX_KOSPI)
    print("\n매도 요청 결과 (삼성전자 10주 @ 67,900원, KOSPI):")
    for k in ("type", "market", "price", "amount", "msg", "state"):
        print(f"  {k:<12}: {r2[k]}")
    print(f"  매도 금액    : {sell_total:,}원")
    print(f"  수수료       : {sell_fee:,}원 ({SELL_COMMISSION*100:.3f}%)")
    print(f"  거래세       : {sell_tax:,}원 ({SELL_TAX_KOSPI*100:.2f}%)")
    print(f"  실수령       : {sell_total - sell_fee - sell_tax:,}원")
    print(f"  잔고         : {test_trader.balance:,} KRW")
    print(f"  손익         : {test_trader.balance - 5_000_000:+,} KRW")

except Exception as e:
    nb_log.flush_on_error("트레이더 테스트 실패")
    raise


## 섹션 6 — 트레이딩 루프 (동기 실행)

**스레딩 없이 5단계를 직접 호출**

```
Step 1  KosdaqRisingDataProvider.get_info()      → trading_info
Step 2  StrategyStockRising.update_trading_info() → 랭킹/현재가 캐시 갱신
Step 3  StrategyStockRising.get_request()         → buy/sell 요청 (호가단위 가격, 정수 주)
Step 4  MockStockTrader.send_request(reqs, cb)    → 즉시 체결 (수수료+거래세)
Step 5  (cb 내) StrategyStockRising.update_result() → 잔고/보유 반영
```

> **장 외 시간 주의**: DEMO_MODE=False 이고 장 외 시간이면 분봉 데이터가 없어
> get_info() 가 빈 리스트를 반환할 수 있습니다.


In [ ]:
def _execute_trading_sync(dp, strategy, trader):
    """트레이딩 루프 1턴을 동기적으로 실행한다."""
    trading_info = dp.get_info()                         # Step 1
    strategy.update_trading_info(trading_info)           # Step 2
    requests_out = strategy.get_request()                # Step 3

    results = []
    if requests_out:
        def _cb(result):
            strategy.update_result(result)               # Step 5
            results.append(result)
        trader.send_request(requests_out, _cb)           # Step 4

    return trading_info, requests_out, results


# ── 1턴 단계별 실행 ──────────────────────────────────────────────────────────
print("=" * 64)
print(" 트레이딩 루프 1턴 — 단계별 실행")
print("=" * 64)

nb_log.clear()
try:
    strategy1 = StrategyStockRising()
    strategy1.initialize(INITIAL_BUDGET, min_price=MIN_PRICE)
    trader1   = MockStockTrader(initial_budget=INITIAL_BUDGET)
    dp1       = KosdaqRisingDataProvider(
        app_key=APP_KEY, app_secret=APP_SECRET,
        access_token=access_token,
        min_price=MIN_PRICE, max_price=MAX_PRICE,
        candle_count=CANDLE_COUNT, top_n=TOP_N,
        demo_mode=DEMO_MODE,
        demo_stocks=DEMO_STOCK_LIST if DEMO_MODE else None,
    )

    # Step 1
    print("\n[Step 1] KosdaqRisingDataProvider.get_info() 호출")
    t0 = time.time()
    trading_info = dp1.get_info()
    elapsed = time.time() - t0
    print(f"  반환 종목: {len(trading_info)}개  ({elapsed:.2f}초)")
    if trading_info:
        for item in trading_info[:3]:
            tick = calc_tick_size(item['closing_price'])
            print(
                f"    [{item['market_type']}] {item['name']}({item['market']}) "
                f"종가={item['closing_price']:,}원 "
                f"호가={tick:,}원 "
                f"상승={item['rise_ratio']*100:.4f}%"
            )

    # Step 2
    print("\n[Step 2] Strategy.update_trading_info(info) 호출")
    strategy1.update_trading_info(trading_info)
    print(f"  갱신된 랭킹: {strategy1.rankings[:5]}")

    # Step 3
    print("\n[Step 3] Strategy.get_request() 호출")
    reqs1 = strategy1.get_request()
    if reqs1:
        print(f"  생성된 요청: {len(reqs1)}개")
        for req in reqs1:
            print(
                f"    [{req['type'].upper()}] {req.get('name','')}({req['market']})  "
                f"가격={req['price']:,}원  수량={req['amount']}주  "
                f"금액={req['price']*req['amount']:,}원"
            )
    else:
        print("  요청 없음")

    # Step 4 + 5
    print("\n[Step 4+5] Trader.send_request() + callback → update_result()")
    if reqs1:
        res1 = []
        def _cb1(result):
            strategy1.update_result(result)
            res1.append(result)
            status = "성공" if result.get("msg") == "success" else result.get("msg")
            print(
                f"  callback → [{result['type'].upper()}] "
                f"{result.get('name','')}({result.get('market','')})  "
                f"{result['price']:,}원 × {result['amount']}주  {status}"
            )
        trader1.send_request(reqs1, _cb1)
        print(f"\n[최종 상태]  잔고={strategy1.balance:,}  "
              f"보유={[(k, v['shares']+'주') for k,v in strategy1.holdings.items()]}")
    else:
        print("  요청 없어 Step 4/5 생략")

    print("\n1턴 완료")

except Exception as e:
    nb_log.flush_on_error("1턴 실행 실패")
    raise


In [ ]:
# ── [Step 3 상세] 호가 단위 반영 확인 ─────────────────────────────────────
if trading_info:
    print("호가 단위 적용 확인 (상위 5종목):")
    print(f"  {'종목':<12}{'현재가':>10}  {'호가단위':>8}  {'주문가':>10}  {'예상수량':>8}")
    print("  " + "-" * 58)
    for item in trading_info[:5]:
        raw_price   = item['closing_price']
        tick        = calc_tick_size(raw_price)
        order_price = floor_to_tick(raw_price)
        shares      = int(MAX_BUY_AMOUNT / order_price / (1 + BUY_COMMISSION))
        shares      = max(1, shares)
        print(
            f"  {item['name']:<12}{raw_price:>10,}원  "
            f"{tick:>7,}원  {order_price:>10,}원  {shares:>7,}주"
        )
else:
    print("상승 종목이 없어 호가 단위 확인 생략")


In [ ]:
print(f"{N_TURNS}턴 시뮬레이션 시작")
print("=" * 72)

nb_log.clear()

strategy_sim = StrategyStockRising()
strategy_sim.initialize(INITIAL_BUDGET, min_price=MIN_PRICE)
trader_sim   = MockStockTrader(initial_budget=INITIAL_BUDGET)
dp_sim       = KosdaqRisingDataProvider(
    app_key=APP_KEY, app_secret=APP_SECRET,
    access_token=access_token,
    min_price=MIN_PRICE, max_price=MAX_PRICE,
    candle_count=CANDLE_COUNT, top_n=TOP_N,
    demo_mode=DEMO_MODE,
    demo_stocks=DEMO_STOCK_LIST if DEMO_MODE else None,
)

turn_records = []

try:
    for turn in range(1, N_TURNS + 1):
        ts = datetime.now().strftime("%H:%M:%S")
        print(f"\n--- Turn {turn}/{N_TURNS}  ({ts}) ---")

        info, reqs, results = _execute_trading_sync(dp_sim, strategy_sim, trader_sim)

        # 보유 자산 평가액 (현재가 기준)
        holdings_val = sum(
            h["current_price"] * h["shares"]
            for h in strategy_sim.holdings.values()
        )
        total_asset = strategy_sim.balance + holdings_val

        # 체결 요약
        executed = [
            f"[{r['type'].upper()}]{r.get('name',r.get('market','?'))}×{r['amount']}주"
            for r in results if r.get("msg") == "success"
        ]

        turn_records.append({
            "turn":         turn,
            "time":         ts,
            "balance":      strategy_sim.balance,
            "holdings_val": round(holdings_val),
            "total_asset":  round(total_asset),
            "n_holdings":   len(strategy_sim.holdings),
            "n_rising":     len(info),
            "n_requests":   len(reqs) if reqs else 0,
        })

        print(
            f"  상승 종목: {len(info):2d}  |  요청: {len(reqs) if reqs else 0}  |  "
            f"체결: {len(executed)}건  |  보유: {len(strategy_sim.holdings)}종목  |  "
            f"잔고: {strategy_sim.balance:,}  |  총자산: {round(total_asset):,} KRW"
        )
        if executed:
            print(f"  체결 내역: {' | '.join(executed)}")

        if not DEMO_MODE and turn < N_TURNS:
            print("  5초 후 다음 턴...")
            time.sleep(5)

except Exception as e:
    nb_log.flush_on_error("다수 턴 시뮬레이션 실패")
    raise

print(f"\n시뮬레이션 완료 ({N_TURNS}턴)")


## 섹션 7 — 결과 분석 및 시각화


In [ ]:
if trader_sim.trade_log:
    df_trades = pd.DataFrame(trader_sim.trade_log)
    # 포맷 변환
    df_display = df_trades.copy()
    df_display["price"]  = df_display["price"].apply(lambda x: f"{x:,}")
    df_display["total"]  = df_display["total"].apply(lambda x: f"{x:,}")
    df_display["fee"]    = df_display["fee"].apply(lambda x: f"{x:,}")
    df_display["tax"]    = df_display["tax"].apply(lambda x: f"{x:,}")

    print("체결 내역:")
    display(
        df_display[["time", "type", "name", "code", "market",
                    "price", "shares", "total", "fee", "tax"]]
        .to_string(index=False)
    )

    # 수수료·거래세 합계
    total_fee = df_trades["fee"].sum()
    total_tax = df_trades["tax"].sum()
    print(f"\n총 수수료: {total_fee:,}원  |  총 거래세: {total_tax:,}원  "
          f"|  총 비용: {total_fee+total_tax:,}원")
else:
    print("체결 내역 없음")


In [ ]:
# ── 최종 수익률 요약 ─────────────────────────────────────────────────────────
holdings_final_val = sum(
    h["current_price"] * h["shares"]
    for h in strategy_sim.holdings.values()
)
total_final  = strategy_sim.balance + holdings_final_val
profit       = total_final - INITIAL_BUDGET
return_pct   = profit / INITIAL_BUDGET * 100

# 비용 계산
if trader_sim.trade_log:
    df_tl       = pd.DataFrame(trader_sim.trade_log)
    total_cost  = df_tl["fee"].sum() + df_tl["tax"].sum()
    total_buys  = len(df_tl[df_tl["type"] == "buy"])
    total_sells = len(df_tl[df_tl["type"] == "sell"])
else:
    total_cost = total_buys = total_sells = 0

print("=" * 64)
print(" 최종 결과 요약")
print("=" * 64)
print(f"  초기 예산        : {INITIAL_BUDGET:>13,} KRW")
print(f"  최종 현금 잔고   : {strategy_sim.balance:>13,} KRW")
print(f"  보유 자산 평가   : {holdings_final_val:>13,.0f} KRW")
print(f"  총 자산 평가     : {total_final:>13,.0f} KRW")
print(f"  손익             : {profit:>+13,.0f} KRW")
print(f"  수익률           : {return_pct:>+13.4f} %")
print(f"  총 비용(수수료+세): {total_cost:>13,.0f} KRW")
print(f"  매수 체결        : {total_buys}건  |  매도 체결: {total_sells}건")

if strategy_sim.holdings:
    print(f"\n  현재 보유 종목 ({len(strategy_sim.holdings)}개):")
    print(f"  {'종목명':<10} {'코드':>8} {'시장':>6} {'수량':>6} {'평균가':>10} "
          f"{'현재가':>10} {'평가손익':>12} {'수익률':>8}")
    print("  " + "-" * 80)
    for code, h in strategy_sim.holdings.items():
        val  = h["current_price"] * h["shares"]
        cost = h["avg_price"]     * h["shares"]
        pnl  = val - cost
        pnl_pct = pnl / cost * 100 if cost > 0 else 0
        print(
            f"  {h.get('name', code):<10} {code:>8} {h.get('market_type','?'):>6} "
            f"{h['shares']:>5}주 {h['avg_price']:>10,} {h['current_price']:>10,} "
            f"{pnl:>+12,} {pnl_pct:>+7.2f}%"
        )
else:
    print("\n  현재 보유 종목 없음")


In [ ]:
if turn_records:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle(
        f"StrategyStockRising 시뮬레이션 — Kiwoom REST API ({N_TURNS}턴)",
        fontsize=13, fontweight="bold"
    )

    turns = [r["turn"] for r in turn_records]
    fmt_krw = mticker.FuncFormatter(lambda x, _: f"{x:,.0f}")

    # 1. 총 자산 변화
    ax1 = axes[0, 0]
    ax1.plot(turns, [r["total_asset"] for r in turn_records],
             "b-o", linewidth=2, markersize=5, label="총 자산")
    ax1.axhline(INITIAL_BUDGET, color="gray", linestyle="--",
                linewidth=1, label="초기 예산")
    ax1.set_title("총 자산 변화 (KRW)")
    ax1.set_xlabel("턴")
    ax1.yaxis.set_major_formatter(fmt_krw)
    ax1.grid(alpha=0.3)
    ax1.legend(fontsize=9)

    # 2. 현금 잔고 vs 보유 자산 누적 영역
    ax2 = axes[0, 1]
    ax2.stackplot(
        turns,
        [r["balance"] for r in turn_records],
        [r["holdings_val"] for r in turn_records],
        labels=["현금 잔고", "보유 주식 평가"],
        colors=["#4CAF50", "#2196F3"], alpha=0.75,
    )
    ax2.set_title("현금 vs 보유 주식 평가액")
    ax2.set_xlabel("턴")
    ax2.yaxis.set_major_formatter(fmt_krw)
    ax2.legend(loc="upper left", fontsize=9)
    ax2.grid(alpha=0.3)

    # 3. 탐지된 지속 상승 종목 수
    ax3 = axes[1, 0]
    ax3.bar(turns, [r["n_rising"] for r in turn_records],
            color="#FF9800", alpha=0.8, edgecolor="white")
    ax3.set_title("턴별 탐지된 지속 상승 종목 수")
    ax3.set_xlabel("턴")
    ax3.set_ylabel("종목 수")
    ax3.grid(axis="y", alpha=0.3)

    # 4. 보유 종목 수 + 요청 건수
    ax4 = axes[1, 1]
    ax4.plot(turns, [r["n_holdings"] for r in turn_records],
             "r-s", linewidth=2, markersize=6, label="보유 종목 수")
    ax4.bar(turns, [r["n_requests"] for r in turn_records],
            alpha=0.35, color="purple", label="주문 요청 건수")
    ax4.set_title("보유 종목 수 / 주문 요청 건수")
    ax4.set_xlabel("턴")
    ax4.set_yticks(range(0, strategy_sim.MAX_BUY_COUNT + 3))
    ax4.legend(fontsize=9)
    ax4.grid(alpha=0.3)

    plt.tight_layout()
    os.makedirs("output", exist_ok=True)
    plt.savefig("output/kw_s7_simulation_result.png", dpi=120,
                bbox_inches="tight")
    plt.show()
    print("차트 저장: output/kw_s7_simulation_result.png")
else:
    print("시각화할 데이터 없음")
